# ParkCast SF — Curb & Transit Features

Four per-block features built from previously-unused JSON files in `data/`:

- `rpp_parcels_200m` — count of parcels in a Residential Permit Parking zone within 200m. High values mark residential territory where non-RPP drivers are ticketed, pushing demand to adjacent non-RPP blocks.
- `muni_stops_200m` — count of MUNI bus/rail stops within 200m. Inverse demand proxy: blocks adjacent to transit have lower drive-and-park demand.
- `shuttle_stops_200m` — count of approved tech-shuttle stops within 200m. Signals tech-worker commute corridors (Mission, SoMa, Noe) where private shuttles compete with parking demand.
- `disabled_signs_50m` — count of (H/C SYM) accessible-parking signs within 50m. These are not general-use spaces, so `total_spaces` overcounts capacity on blocks with many of them.

**Output:** `data/block_curb_transit_features.csv` keyed on `cnn`.

## Imports

In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

## Configuration

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(PROJECT_DIR, "app", "models")

MASTER    = os.path.join(MODEL_DIR, "master_blocks.parquet")
RPP       = os.path.join(DATA_DIR, "rpp_parcels.json")
MUNI      = os.path.join(DATA_DIR, "muni_stops.json")
SHUTTLE   = os.path.join(DATA_DIR, "shuttle_stops.json")
BLUE_CURB = os.path.join(DATA_DIR, "blue_curb.json")
OUT       = os.path.join(DATA_DIR, "block_curb_transit_features.csv")

# Radius per feature family. Transit / residential — wider; disabled-parking
# signs — tight (they affect only the block they're on).
RADIUS_TRANSIT  = 200.0
RADIUS_RPP      = 200.0
RADIUS_DISABLED = 50.0

# Equirectangular projection constants (same as build_poi_features.ipynb).
LAT0 = 37.77
M_PER_DEG_LAT = 111_000.0
M_PER_DEG_LON = 111_000.0 * np.cos(np.radians(LAT0))

def to_xy(lat, lon):
    y = (lat - LAT0) * M_PER_DEG_LAT
    x = (lon + 122.4194) * M_PER_DEG_LON
    return x, y

## `load_point_data()`

MUNI / shuttle / blue-curb files are lists of dicts with `latitude` and `longitude`. Filter blue-curb to disabled-parking signs only (legend starts with `(H/C SYM)`) — the rest are time-limit / bike-wayfinding signs, not capacity-reducing.

In [3]:
def load_point_data(path, filter_fn=None):
    with open(path) as f:
        data = json.load(f)
    rows = []
    for r in data:
        if filter_fn is not None and not filter_fn(r):
            continue
        lat = r.get("latitude")
        lon = r.get("longitude")
        # Some DataSF dumps put coords inside a GeoJSON Point under `shape`.
        if lat is None or lon is None:
            shape = r.get("shape") or {}
            if shape.get("type") == "Point":
                coords = shape.get("coordinates") or []
                if len(coords) == 2:
                    lon, lat = coords[0], coords[1]
        if lat is None or lon is None:
            continue
        try:
            rows.append((float(lat), float(lon)))
        except (TypeError, ValueError):
            continue
    return pd.DataFrame(rows, columns=["lat", "lon"])

def is_disabled_sign(row):
    legend = (row.get("legend") or "").upper()
    return "H/C SYM" in legend

## `load_rpp_centroids()`

RPP parcels are MultiPolygons. We use the centroid of the first ring of the first polygon as a representative point — parcels are small enough (single lots) that this is accurate to well under our 200m radius.

In [4]:
def load_rpp_centroids():
    with open(RPP) as f:
        data = json.load(f)
    rows = []
    for r in data:
        if not r.get("rppeligib"):
            continue
        shape = r.get("shape")
        if not shape or shape.get("type") != "MultiPolygon":
            continue
        coords = shape.get("coordinates")
        if not coords or not coords[0] or not coords[0][0]:
            continue
        ring = coords[0][0]  # first ring of first polygon
        lons = [p[0] for p in ring]
        lats = [p[1] for p in ring]
        rows.append((sum(lats) / len(lats), sum(lons) / len(lons)))
    return pd.DataFrame(rows, columns=["lat", "lon"])

## `count_within()`

Generic per-block count-within-radius using a cKDTree in the local-metric projection.

In [5]:
def count_within(blocks, points, radius_m):
    if len(points) == 0:
        return np.zeros(len(blocks), dtype=int)
    bx, by = to_xy(blocks["lat"].values, blocks["lon"].values)
    px, py = to_xy(points["lat"].values, points["lon"].values)
    tree = cKDTree(np.column_stack([px, py]))
    counts = tree.query_ball_point(np.column_stack([bx, by]), r=radius_m)
    return np.array([len(c) for c in counts], dtype=int)

## Pipeline

In [6]:
print("=" * 60)
print("ParkCast SF — Curb & Transit Features")
print("=" * 60)

master = pd.read_parquet(MASTER)
print(f"Master blocks: {len(master):,}")

print("Loading point datasets...")
rpp      = load_rpp_centroids()
muni     = load_point_data(MUNI)
shuttle  = load_point_data(SHUTTLE)
disabled = load_point_data(BLUE_CURB, filter_fn=is_disabled_sign)
print(f"  RPP parcels      : {len(rpp):,}")
print(f"  MUNI stops       : {len(muni):,}")
print(f"  Shuttle stops    : {len(shuttle):,}")
print(f"  Disabled signs   : {len(disabled):,}")

feats = pd.DataFrame({"cnn": master["cnn"].values})
feats["rpp_parcels_200m"]    = count_within(master, rpp,      RADIUS_RPP)
feats["muni_stops_200m"]     = count_within(master, muni,     RADIUS_TRANSIT)
feats["shuttle_stops_200m"]  = count_within(master, shuttle,  RADIUS_TRANSIT)
feats["disabled_signs_50m"]  = count_within(master, disabled, RADIUS_DISABLED)

feats.to_csv(OUT, index=False)
print(f"\nSaved → {OUT}")
print("Summary per feature:")
print(feats.drop(columns=["cnn"]).describe().round(1).to_string())
print("=" * 60)

ParkCast SF — Curb & Transit Features
Master blocks: 12,242
Loading point datasets...


  RPP parcels      : 83,359
  MUNI stops       : 3,260
  Shuttle stops    : 164
  Disabled signs   : 1,181

Saved → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/data/block_curb_transit_features.csv
Summary per feature:
       rpp_parcels_200m  muni_stops_200m  shuttle_stops_200m  disabled_signs_50m
count           12242.0          12242.0             12242.0             12242.0
mean              121.2              4.4                 0.3                 0.1
std               184.3              3.5                 0.7                 0.5
min                 0.0              0.0                 0.0                 0.0
25%                 0.0              2.0                 0.0                 0.0
50%                24.0              4.0                 0.0                 0.0
75%               220.0              6.0                 0.0                 0.0
max              2491.0             22.0                 6.0                16.0
